In [1]:
# --------------------------------------------------
# 0. Import
# --------------------------------------------------

import pandas as pd

In [2]:
# --------------------------------------------------
# 1. Load files
# --------------------------------------------------

ml_path = "ml_gold_summary_match_nonmatch.csv"
cnn_path = "cnn_gold_summary_match_nonmatch.csv"
bert_path = "bert_scibert_gold_summary_match_nonmatch.csv"

ml_df = pd.read_csv(ml_path, encoding="utf-8-sig")
cnn_df = pd.read_csv(cnn_path, encoding="utf-8-sig")
bert_df = pd.read_csv(bert_path, encoding="utf-8-sig")

In [3]:
# --------------------------------------------------
# 2. Standardize file structure
# --------------------------------------------------

def standardize_summary(df):
    # Remove Dataset column if it exists
    if "Dataset" in df.columns:
        df = df.drop(columns=["Dataset"])
    
    # Keep only Match and NonMatch rows
    df = df[df["Class"].isin(["Match", "NonMatch"])].copy()
    
    # Keep only necessary columns
    df = df[["Model", "Class", "Precision", "Recall", "F1"]]
    
    return df


ml_clean = standardize_summary(ml_df)
cnn_clean = standardize_summary(cnn_df)
bert_clean = standardize_summary(bert_df)

# Combine all models
combined_df = pd.concat(
    [ml_clean, cnn_clean, bert_clean],
    ignore_index=True
)

print(combined_df)

                Model     Class  Precision    Recall        F1
0  LogisticRegression  NonMatch   0.872585  0.590459  0.704320
1  LogisticRegression     Match   0.217421  0.568905  0.314607
2        RandomForest  NonMatch   0.912538  0.637809  0.750832
3        RandomForest     Match   0.277151  0.694346  0.396169
4                 CNN  NonMatch   0.866993  0.626502  0.727385
5                 CNN     Match   0.217617  0.519435  0.306729
6                BERT  NonMatch   0.931283  0.684806  0.789249
7                BERT     Match   0.321673  0.747350  0.449761
8             SciBERT  NonMatch   0.926521  0.672792  0.779529
9             SciBERT     Match   0.309471  0.733216  0.435239


In [4]:
# --------------------------------------------------
# 3. Set model order and display names
# --------------------------------------------------

model_order = [
    "LogisticRegression",
    "RandomForest",
    "CNN",
    "BERT",
    "SciBERT"
]

model_name_map = {
    "LogisticRegression": "Logistic Regression",
    "RandomForest": "Random Forest",
    "CNN": "CNN",
    "BERT": "BERT",
    "SciBERT": "SciBERT"
}

combined_df["Model_Display"] = combined_df["Model"].replace(model_name_map)

display_order = [model_name_map[m] for m in model_order]

combined_df["Model_Display"] = pd.Categorical(
    combined_df["Model_Display"],
    categories=display_order,
    ordered=True
)

combined_df = combined_df.sort_values(["Model_Display", "Class"]).reset_index(drop=True)

In [5]:
# --------------------------------------------------
# 4. Build publication-ready tables
# --------------------------------------------------

def format_value(x, digits=4):
    return f"{float(x):.{digits}f}"


def build_class_table(df, class_label):
    # Select Match or NonMatch rows
    temp = df[df["Class"] == class_label].copy()
    
    # Sort by predefined model order
    temp = temp.sort_values("Model_Display")
    
    # Build final table
    table = pd.DataFrame({
        "Model": temp["Model_Display"].astype(str),
        "Precision": temp["Precision"].apply(format_value),
        "Recall": temp["Recall"].apply(format_value),
        "F1": temp["F1"].apply(format_value)
    })
    
    return table


match_table = build_class_table(combined_df, "Match")
nonmatch_table = build_class_table(combined_df, "NonMatch")

print("Positive Pair (Match) Results")
display(match_table)

print("Negative Pair (NonMatch) Results")
display(nonmatch_table)

Positive Pair (Match) Results


,Model,Precision,Recall,F1
0,Logistic Regression,0.2174,0.5689,0.3146
2,Random Forest,0.2772,0.6943,0.3962
4,CNN,0.2176,0.5194,0.3067
6,BERT,0.3217,0.7473,0.4498
8,SciBERT,0.3095,0.7332,0.4352


Negative Pair (NonMatch) Results


,Model,Precision,Recall,F1
1,Logistic Regression,0.8726,0.5905,0.7043
3,Random Forest,0.9125,0.6378,0.7508
5,CNN,0.8670,0.6265,0.7274
7,BERT,0.9313,0.6848,0.7892
9,SciBERT,0.9265,0.6728,0.7795


In [6]:
# --------------------------------------------------
# 5. Save output tables
# --------------------------------------------------

match_table.to_csv(
    "gold_transfer_positive_match_table.csv",
    index=False,
    encoding="utf-8-sig"
)

nonmatch_table.to_csv(
    "gold_transfer_negative_nonmatch_table.csv",
    index=False,
    encoding="utf-8-sig"
)

combined_df.to_csv(
    "gold_transfer_combined_summary_clean.csv",
    index=False,
    encoding="utf-8-sig"
)